<a href="https://colab.research.google.com/github/alvarobuchen/alvarobuchen-DataScience-GenAI-Submissions/blob/main/assignments/6_02_DNN_101_COMPLETED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1xqQczl0FG-qtNA2_WQYuWePW9oU8irqJ)

# 6.02 Dense Neural Network (with PyTorch)
This will expand on our logistic regression example and take us through building our first neural network. If you haven't already, be sure to check (and if neccessary) switch to GPU processing by clicking Runtime > Change runtime type and selecting GPU. We can test this has worked with the following code:

In [1]:
import torch

# Check for GPU availability
print("Num GPUs Available: ", torch.cuda.device_count())

Num GPUs Available:  1


Hopefully your code shows you have 1 GPU available! Next let's get some data. We'll start with another in-built dataset:

In [2]:
# upload an in-built Python (OK semi-in-built) dataset
from sklearn.datasets import load_diabetes

import pandas as pd
import numpy as np

# import the data
data = load_diabetes()
data

{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]], shape=(442, 10)),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142.

We are working on a regression problem, with "structured" data which has already been cleaned and normalised. We can skip the usual cleaning/engineering steps. However, we do need to get the data into PyTorch:

In [3]:
# Convert data to PyTorch tensors
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1) # Reshape y to be a column vector

Now our data is stored in tensors we can do train/test splitting as before (in fact we can use sklearn as before):

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


Now we can set up our batches for training. As we have a nice round 400 let's go with batches of 50 (8 batches in total). We'll also seperate the features and labels:

In [5]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

Now its time to build our model. We'll keep it simple ... a model with an input layer of 10 features and then 2x _Dense_ (fully connected) layers each with 5 neurons and ReLU activation. Our output layer will be size=1 given this is a regression problem and we want a single value output per prediction.

This will be easier to understand if you have read through the logistic regression tutorial.

In [6]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer will be a linear layer that has 5x neurons
            # (5x sets of linear regression)
            # the layer takes the 10 features as input (i.e. 10, 5)
            nn.Linear(10, 5),

            nn.ReLU(), # ReLU activation

            # second linear layer again has 5 neurons
            # this time taking the input as the output of the last layer
            # (which had 5x neurons)
            nn.Linear(5, 5),

            nn.ReLU(), # ReLU again

            # last linear layer takes the output from the previous 5 neurons
            # this time its a single output with no activation
            # i.e. this is the predicitons (regression)
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

As before we need to create a model object, specify the loss (criterion) and an optimiser (which we cover next week):

In [7]:
import torch.optim as optim

# Initialize the model, loss function, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss() # MSE loss function
optimiser = optim.Adam(model.parameters(), lr=0.001)

Now we can train the model. Again, the logistic regression tutorial (6.01) may help you undertstand this:

In [8]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop (example - you'll likely want to add more epochs)
epochs = 100 # 100 epochs

for epoch in range(epochs):
  # use the train_loader to pass the inputs (x) and targets (y)
  for inputs, targets in train_loader:
    # pass to the GPU (hopefully)
    inputs, targets = inputs.to(device), targets.to(device)

    # pass model to GPU as well
    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/100], Loss: 27389.1387
Epoch [10/100], Loss: 30399.6934
Epoch [10/100], Loss: 31265.5996
Epoch [10/100], Loss: 33630.0586
Epoch [10/100], Loss: 27826.5586
Epoch [10/100], Loss: 33041.2578
Epoch [10/100], Loss: 25109.2285
Epoch [10/100], Loss: 8771.6895
Epoch [20/100], Loss: 30663.6152
Epoch [20/100], Loss: 31133.4551
Epoch [20/100], Loss: 30275.3438
Epoch [20/100], Loss: 21205.3691
Epoch [20/100], Loss: 33996.625
Epoch [20/100], Loss: 34974.9961
Epoch [20/100], Loss: 24714.2051
Epoch [20/100], Loss: 31740.7617
Epoch [30/100], Loss: 27224.2539
Epoch [30/100], Loss: 30317.5449
Epoch [30/100], Loss: 26205.9453
Epoch [30/100], Loss: 29745.9785
Epoch [30/100], Loss: 35416.1602
Epoch [30/100], Loss: 30948.6484
Epoch [30/100], Loss: 26848.2285
Epoch [30/100], Loss: 25771.3965
Epoch [40/100], Loss: 31558.7695
Epoch [40/100], Loss: 32227.7539
Epoch [40/100], Loss: 26592.1973
Epoch [40/100], Loss: 29747.2734
Epoch [40/100], Loss: 25515.6602
Epoch [40/100], Loss: 30340.7109
Epoch [40/10

We can see loss is significantly lower at the end than it was at the start. However, it is also bouncing around a little still which suggests the model needs more training (100 epochs is not a lot in deep learning terms). However, let's evaluate as before:

In [9]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")

Average MSE on test set: 22536.4072265625


MSE looks expected given training (no obvious sign of overfitting). However, we probably can get better results with tuning and more epochs.

Let's run the loop again a little differently to collect the predicted values (y_hat) and actuals (y) and add them to a dataset for comparions:

In [10]:
# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df

,Predicted,Actual
0,13.926718,219.0
1,13.072901,70.0
2,13.702515,202.0
3,16.896158,230.0
4,13.254475,111.0
...,...,...
84,11.861591,153.0
85,10.951618,98.0
86,9.891514,37.0
87,10.312284,63.0


Side-by-side, they don't look great. Can you improve them?

<br><br>

## EXERCISE #1
Try increasing the number of epochs to 1,000 (when the model is fairly well trained then the results printed for each 10x epochs will be fairly stable and not change much). Does this give better results?

<br><br>

## EXERCISE #2 (optional)
Try experimenting with the architecture (number of neurons and/or number of layers). Can we reach an optimal architecture?

In [15]:
# EXERCISE #1
torch.manual_seed(42) # fix the random starting weights so results are repeatable
model = DiabetesModel()
criterion = nn.MSELoss()
optimiser = optim.Adam(model.parameters(), lr=0.001)
model.to(device)

epochs = 1000
for epoch in range(epochs):
    model.train()
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimiser.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimiser.step()
    # print once every 50 epochs (last batch loss) to keep the output readable
    if (epoch+1) % 50 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

# Evaluation - same as before
model.eval()
mse_values = []
predictions = []
actuals = []
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        mse_values.append(criterion(outputs, targets).item())
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

print(f"Average MSE on test set: {np.mean(mse_values)}")
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df.head(15)


Epoch [50/1000], Loss: 22986.125
Epoch [100/1000], Loss: 12714.1826
Epoch [150/1000], Loss: 12779.377
Epoch [200/1000], Loss: 3944.5476
Epoch [250/1000], Loss: 2895.1953
Epoch [300/1000], Loss: 3042.3042
Epoch [350/1000], Loss: 2832.2773
Epoch [400/1000], Loss: 1842.0409
Epoch [450/1000], Loss: 10447.5391
Epoch [500/1000], Loss: 2892.4766
Epoch [550/1000], Loss: 5949.1035
Epoch [600/1000], Loss: 3807.3125
Epoch [650/1000], Loss: 3277.3047
Epoch [700/1000], Loss: 1849.7197
Epoch [750/1000], Loss: 3402.022
Epoch [800/1000], Loss: 2837.738
Epoch [850/1000], Loss: 4909.793
Epoch [900/1000], Loss: 897.623
Epoch [950/1000], Loss: 3297.0261
Epoch [1000/1000], Loss: 3682.0603
Average MSE on test set: 2953.8369140625


,Predicted,Actual
0,163.268661,219.0
1,166.349365,70.0
2,156.468216,202.0
3,279.684875,230.0
4,145.359253,111.0
5,122.791504,84.0
6,225.171326,242.0
7,199.266891,272.0
8,107.446686,94.0
9,127.618576,96.0


In [14]:
# EXERCISE #2
class FlexibleModel(nn.Module):
    def __init__(self, hidden_sizes):
        super(FlexibleModel, self).__init__()
        layers = []
        n_in = 10 # 10 input features
        for n_out in hidden_sizes:
            layers.append(nn.Linear(n_in, n_out)) # Dense (fully connected) layer
            layers.append(nn.ReLU())              # ReLU activation
            n_in = n_out
        layers.append(nn.Linear(n_in, 1))         # single regression output, no activation
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)

def average_mse(model, loader):
    model.eval()
    mse_values = []
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            mse_values.append(criterion(model(inputs), targets).item())
    return np.mean(mse_values)

def train_and_evaluate(hidden_sizes, epochs=1000):
    torch.manual_seed(42) # same starting point for every architecture
    model = FlexibleModel(hidden_sizes).to(device)
    optimiser = optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(epochs):
        model.train()
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimiser.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimiser.step()
    return average_mse(model, train_loader), average_mse(model, test_loader)

architectures = [
    [5, 5],        # original model (baseline)
    [10],          # one wider layer
    [10, 10],      # two layers, more neurons
    [20, 20],
    [50, 50],
    [10, 10, 10],  # three layers
    [20, 10, 5],   # "funnel" shape
    [50, 50, 50],
]

results = []
for arch in architectures:
    train_mse, test_mse = train_and_evaluate(arch)
    print(f"{str(arch):<14} train MSE: {train_mse:9.1f}   test MSE: {test_mse:9.1f}")
    results.append({'Hidden layers': str(arch), 'Train MSE': round(train_mse, 1), 'Test MSE': round(test_mse, 1)})

arch_df = pd.DataFrame(results).sort_values('Test MSE')
arch_df


[5, 5]         train MSE:    3229.3   test MSE:    2953.8
[10]           train MSE:    4570.7   test MSE:    3555.6
[10, 10]       train MSE:    2783.4   test MSE:    2864.1
[20, 20]       train MSE:    3030.8   test MSE:    2861.8
[50, 50]       train MSE:    2859.0   test MSE:    2837.2
[10, 10, 10]   train MSE:   25705.7   test MSE:   23962.7
[20, 10, 5]    train MSE:    3283.6   test MSE:    2853.8
[50, 50, 50]   train MSE:    3106.8   test MSE:    2613.0


,Hidden layers,Train MSE,Test MSE
7,"[50, 50, 50]",3106.8,2613.0
4,"[50, 50]",2859.0,2837.2
6,"[20, 10, 5]",3283.6,2853.8
3,"[20, 20]",3030.8,2861.8
2,"[10, 10]",2783.4,2864.1
0,"[5, 5]",3229.3,2953.8
1,[10],4570.7,3555.6
5,"[10, 10, 10]",25705.7,23962.7
